In [3]:


pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet] [fastparquet]
Note: you may need to restart the kernel to use updated packages.


In [26]:
import numpy as np
import pandas as pd

#importation fichier .parquet
httrain= pd.read_parquet("history_toulouse_train.parquet")
httest=pd.read_parquet("history_toulouse_test.parquet")
httest

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00002 - LAFAYETTE,2026-07-31 23:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - POMME,2025-01-05 05:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PASTEUR - LAVOISIER,2025-08-05 10:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
#ajout serie(colonne) heure en fomrat datetime
t=httrain.index.get_level_values("commit_at")

httrain["jour"]=t.dayofweek
httrain["heure"]=t.hour
httrain["minute"]=t.minute


httrain

bikes  stands  jour  heure  \
station                  commit_at                                         
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9     5     12   
                         2023-08-05 12:30:00     10       9     5     12   
                         2023-08-05 12:45:00     10       9     5     12   
                         2023-08-05 13:00:00     11       8     5     13   
                         2023-08-05 13:15:00     11       8     5     13   
...                                             ...     ...   ...    ...   
TEST BEE                 2026-02-12 15:00:00      0       3     3     15   
                         2026-02-12 15:15:00      0       3     3     15   
                         2026-02-12 15:30:00      0       3     3     15   
                         2026-02-12 15:45:00      0       3     3     15   
                         2026-02-12 16:00:00      0       3     3     16   

                                              minute  
station                  commit_at                    
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00      15  
                         2023-08-05 12:30:00      30  
                         2023-08-05 12:45:00      45  
                         2023-08-05 13:00:00       0  
                         2023-08-05 13:15:00      15  
...                                              ...  
TEST BEE                 2026-02-12 15:00:00       0  
                         2026-02-12 15:15:00      15  
                         2026-02-12 15:30:00      30  
                         2026-02-12 15:45:00      45  
                         2026-02-12 16:00:00       0  

[40012622 rows x 5 columns]

In [28]:
#calcul moyenne des velo par quart d'heure
profil_moyen= httrain.groupby(["station","jour", "heure", "minute"])["bikes"].mean()
     
profil_moyen


station                   jour  heure  minute
00001 - POIDS DE L'HUILE  0     0      0         2.743590
                                       15        2.641026
                                       30        2.467949
                                       45        2.365385
                                1      0         2.243590
                                                   ...   
TEST BEE                  3     15     0         0.000000
                                       15        0.000000
                                       30        0.000000
                                       45        0.000000
                                16     0         0.000000
Name: bikes, Length: 532897, dtype: float64

In [14]:
# ajout des resultat dans httest
target_columns = [
    "bikes_+15min", "bikes_+30min", "bikes_+45min", "bikes_+60min",
    "bikes_+75min", "bikes_+90min", "bikes_+105min", "bikes_+120min"
]

delta_hour = [15, 30, 45, 60, 75, 90, 105, 120]


forecast= pd.to_datetime(httest.index.get_level_values("commit_at"))
stations= httest.index.get_level_values(0)

for columns in range(8):
    
    time_delta= forecast + pd.Timedelta(minutes=delta_hour[columns])
    index_recherche= pd.MultiIndex.from_arrays([
        stations,
        time_delta.day_name().astype(str),
        time_delta.hour.astype(int),
        time_delta.minute.astype(int)
    ], names=["station", "jour" , "heure", "minute"])


    
    moyennes_futures = index_recherche.map(meanbh)
    serie_moyennes = pd.Series(moyennes_futures, index=httest.index)

    httest[target_columns[columns]] =  httest[target_columns[columns]].fillna(serie_moyennes).fillna(0) #.round().astype(int)

print(moyennes_futures)

httest

Index([2.9419354838709677,  4.316129032258065,  7.981132075471698,
        1.837837837837838,  7.648648648648648,  7.826923076923077,
                      nan, 1.9736842105263157,  5.472972972972973,
        4.094594594594595,
       ...
                      0.0,                nan,                0.0,
                     14.0,               12.0,                0.0,
                      nan,                nan,                nan,
                      nan],
      dtype='float64', length=800)


,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,5.554839,5.206452,4.722581,4.432258,4.135484,3.625806,3.193548,2.941935
00002 - LAFAYETTE,2026-07-31 23:45:00,5.690323,5.580645,5.322581,5.180645,4.838710,4.774194,4.477419,4.316129
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11.094340,10.320755,10.962264,10.584906,9.471698,9.094340,8.018868,7.981132
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,6.000000,6.027027,5.027027,4.540541,3.621622,3.486486,2.972973,1.837838
00003 - POMME,2025-01-05 05:45:00,7.175676,7.297297,7.364865,7.418919,7.500000,7.459459,7.378378,7.648649
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [39]:
stations=httest.index.get_level_values("station")
t_test=httest.index.get_level_values("commit_at")

for i in range(15,121,15):
    cible=t_test+pd.Timedelta(minutes=i)
    cles=pd.MultiIndex.from_arrays(
        [stations, cible.dayofweek, cible.hour, cible.minute], 
        names=["stations", "jour", "heure", "minute"],
    )
    httest[f"bikes_+{i}min"]=profil_moyen.reindex(cles).to_numpy()

httest = httest.fillna(httrain["bikes"].mean())
httest.round().astype(int)


,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,6,5,5,4,4,4,3,3
00002 - LAFAYETTE,2026-07-31 23:45:00,6,6,5,5,5,5,4,4
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11,10,11,11,9,9,8,8
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,6,6,5,5,4,3,3,2
00003 - POMME,2025-01-05 05:45:00,7,7,7,7,8,7,7,8
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,7,7,7,7,7,7,7,7
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,7,7,7,7,7,7,7,7


In [35]:
display(httest)

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,5.554839,5.206452,4.722581,4.432258,4.135484,3.625806,3.193548,2.941935
00002 - LAFAYETTE,2026-07-31 23:45:00,5.690323,5.580645,5.322581,5.180645,4.838710,4.774194,4.477419,4.316129
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11.094340,10.320755,10.962264,10.584906,9.471698,9.094340,8.018868,7.981132
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,6.000000,6.027027,5.027027,4.540541,3.621622,3.486486,2.972973,1.837838
00003 - POMME,2025-01-05 05:45:00,7.175676,7.297297,7.364865,7.418919,7.500000,7.459459,7.378378,7.648649
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PASTEUR - LAVOISIER,2025-08-05 10:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
httest.to_parquet(path= "prevision/prediction_fct1.parquet", engine='auto', index=False) 

In [ ]:
hour_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().hour
print(hour_forecast)
minute_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().minute
print(minute_forecast)